# 03 · Segmentación exploratoria de clientes

Ruta de continuación con las seis fases CRISP-DM. Los CSV públicos se cargan desde el repositorio.

## 1. Comprensión del negocio

Buscamos patrones de compra similares que merezcan investigación. Los grupos son exploratorios, no categorías verdaderas ni perfiles para tomar decisiones sobre personas.

## 2. Comprensión de los datos

Usamos fechas y estado de pedidos, identificadores seudónimos para agrupar, precios por línea y categorías de producto. No hay inventario ni cantidad explícita por línea. El [diccionario](../docs/diccionario_datos.md) describe el esquema disponible.


In [ ]:
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_BASE_URL = "https://raw.githubusercontent.com/JuanParias29/ML_MiniWorkshop/main/data"

def load_dataset_csv(filename):
    url = f"{DATA_BASE_URL}/{filename}"
    try:
        return pd.read_csv(url)
    except (HTTPError, URLError) as error:
        raise RuntimeError(
            f"No se pudo cargar {filename} desde {url}. Compruebe la conexión a internet "
            "y confirme que el CSV está publicado en la carpeta main/data del repositorio."
        ) from error
orders = load_dataset_csv("olist_orders_dataset.csv")
customers = load_dataset_csv("olist_customers_dataset.csv")
items = load_dataset_csv("olist_order_items_dataset.csv")
products = load_dataset_csv("olist_products_dataset.csv")
print(f"Pedidos: {len(orders):,}; líneas de artículos: {len(items):,}")


## 3. Preparación de los datos

Agregaremos recencia, frecuencia, gasto en precios de artículos y variedad de categorías por cliente. El gasto no incluye flete. Los identificadores solo permiten agrupar transacciones, no son variables del modelo.


In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)
orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id", how="inner"
)
delivered = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_purchase_timestamp"].notna()
    & orders["customer_unique_id"].notna()
].copy()
if delivered.empty:
    raise ValueError("No hay pedidos entregados con cliente y fecha válida.")
reference_date = delivered["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
order_map = delivered[["order_id", "customer_unique_id", "order_purchase_timestamp"]]
purchases = items.merge(order_map, on="order_id", how="inner").merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left"
)
customer_features = purchases.groupby("customer_unique_id").agg(
    frequency=("order_id", "nunique"),
    spend_brl=("price", "sum"),
    category_variety=("product_category_name", "nunique"),
    last_purchase=("order_purchase_timestamp", "max"),
)
customer_features["recency_days"] = (
    reference_date - customer_features.pop("last_purchase")
).dt.total_seconds() / 86400
feature_columns = ["recency_days", "frequency", "spend_brl", "category_variety"]
customer_features = customer_features[feature_columns].dropna()
print(f"Clientes agregados: {len(customer_features):,}")
display(customer_features.describe().round(1))

## 4. Modelado

K-Means requiere escalar las variables porque utiliza distancias. Usamos cuatro grupos como punto de partida didáctico; ese número no es una verdad descubierta. La semilla fija permite reproducir esta ejecución.


In [ ]:
n_clusters = 4
if len(customer_features) < n_clusters:
    raise ValueError(f"K-Means con {n_clusters} grupos requiere al menos ese número de clientes.")
cluster_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=n_clusters, random_state=42, n_init=10),
)
customer_features["cluster"] = cluster_model.fit_predict(customer_features[feature_columns])
cluster_profile = customer_features.groupby("cluster")[feature_columns].mean().round(1)
display(cluster_profile)

## 5. Evaluación

Una proyección PCA a dos dimensiones ayuda a observar los grupos, pero no conserva toda la información. Los promedios se presentan en las unidades originales para facilitar la interpretación. No hay etiquetas verdaderas ni una evaluación de causalidad.


In [ ]:
scaled_features = cluster_model.named_steps["standardscaler"].transform(
    customer_features[feature_columns]
)
projection = PCA(n_components=2, random_state=42).fit_transform(scaled_features)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    projection[:, 0], projection[:, 1],
    c=customer_features["cluster"], cmap="tab10", alpha=0.65, s=18,
)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("K-Means · proyección PCA exploratoria")
plt.colorbar(scatter, label="Grupo asignado")
plt.show()
print("Perfiles promedio en las unidades originales (no son etiquetas reales):")
display(cluster_profile)

## 6. Despliegue (demostración temporal)

Pruebe características agregadas de ejemplo para ver el grupo asignado. Esto no clasifica a una persona real. La interfaz de Colab es temporal y no es un despliegue de producción.


In [ ]:
import gradio as gr

def cluster_demo(recency_days, frequency, spend_brl, category_variety):
    example = pd.DataFrame(
        [[recency_days, frequency, spend_brl, category_variety]],
        columns=feature_columns,
    )
    group = int(cluster_model.predict(example)[0])
    return f"Grupo exploratorio asignado: {group}. No es una categoría verdadera ni una decisión sobre una persona."

demo = gr.Interface(
    fn=cluster_demo,
    inputs=[
        gr.Number(value=60, minimum=0, label="Recencia (días)"),
        gr.Number(value=2, minimum=1, label="Pedidos observados"),
        gr.Number(value=200, minimum=0, label="Suma de precios (BRL)"),
        gr.Number(value=2, minimum=0, label="Categorías distintas"),
    ],
    outputs=gr.Textbox(label="Resultado exploratorio"),
    title="Demo de K-Means",
    description="Pruebe valores agregados; los grupos dependen de los datos y del escalado.",
)
demo.launch(share=False, inline=True, prevent_thread_lock=True)